In [ ]:
from pathlib import Path

import pandas as pd

trades_path = Path("kxnbagame_trades_all.parquet")
if not trades_path.exists():
    trades_path = Path("..") / "kxnbagame_trades_all.parquet"

trades = pd.read_parquet(trades_path)
trades["created_time"] = pd.to_datetime(trades["created_time"])

trades.head()

In [5]:
import duckdb

with duckdb.connect() as con:
    nba_data = con.execute("""
        SELECT
            event_ticker,
            SUBSTRING(event_ticker, 1, 8)  AS game_date,
            SUBSTRING(event_ticker, 12, 3) AS sport,
            SUBSTRING(event_ticker, 15, 3) AS league,
            SUBSTRING(event_ticker, 18, 3) AS away_team,
            SUBSTRING(event_ticker, 21, 3) AS home_team,
            SPLIT_PART(ticker, '-', 3)     AS team,
            result,
            *
        FROM read_csv_auto('kxnbagame_markets_all.csv')
        WHERE result IN ('yes', 'no')
        LIMIT 1000
    """).df()

nba_data.head()


,event_ticker,game_date,sport,league,away_team,home_team,team,result,ticker,event_ticker_1,...,subtitle,open_time,close_time,settlement_ts,result_1,settlement_value_dollars,volume_fp,last_price_dollars,yes_bid_dollars,no_bid_dollars
0,KXNBAGAME-25APR15ATLORL,KXNBAGAM,5AP,R15,ATL,ORL,ORL,yes,KXNBAGAME-25APR15ATLORL-ORL,KXNBAGAME-25APR15ATLORL,...,None,2025-04-14 08:00:00-10:00,2025-04-15 16:39:09.736516-10:00,2025-04-15 16:47:32.782451-10:00,yes,1.0,199125.0,0.99,0.99,0.00
1,KXNBAGAME-25APR15ATLORL,KXNBAGAM,5AP,R15,ATL,ORL,ATL,no,KXNBAGAME-25APR15ATLORL-ATL,KXNBAGAME-25APR15ATLORL,...,None,2025-04-14 08:00:00-10:00,2025-04-15 16:39:09.736516-10:00,2025-04-15 16:47:32.782451-10:00,no,0.0,185969.0,0.01,0.00,0.99
2,KXNBAGAME-25APR15MEMGSW,KXNBAGAM,5AP,R15,MEM,GSW,MEM,no,KXNBAGAME-25APR15MEMGSW-MEM,KXNBAGAME-25APR15MEMGSW,...,None,2025-04-14 08:00:00-10:00,2025-04-15 18:59:04.802883-10:00,2025-04-15 19:07:27.782601-10:00,no,0.0,319985.0,0.01,0.00,0.99
3,KXNBAGAME-25APR15MEMGSW,KXNBAGAM,5AP,R15,MEM,GSW,GSW,yes,KXNBAGAME-25APR15MEMGSW-GSW,KXNBAGAME-25APR15MEMGSW,...,None,2025-04-14 08:00:00-10:00,2025-04-15 18:59:04.802883-10:00,2025-04-15 19:07:27.782601-10:00,yes,1.0,345417.0,0.99,0.99,0.00
4,KXNBAGAME-25APR16MIACHI,KXNBAGAM,5AP,R16,MIA,CHI,CHI,no,KXNBAGAME-25APR16MIACHI-CHI,KXNBAGAME-25APR16MIACHI,...,None,2025-04-14 09:15:00-10:00,2025-04-16 16:12:34.041146-10:00,2025-04-16 16:20:57.781509-10:00,no,0.0,411138.0,0.01,0.00,0.99


In [ ]:
import duckdb

with duckdb.connect() as con:
    wavg = con.execute("""
        WITH markets AS (
            SELECT
                ticker,
                event_ticker,
                SPLIT_PART(ticker, '-', -1)           AS team,
                CAST(close_time AS TIMESTAMPTZ)        AS close_time,
                result,
                CASE WHEN result = 'yes' THEN 1 ELSE 0 END AS outcome
            FROM read_csv_auto('kxnbagame_markets_all.csv')
            WHERE result IN ('yes', 'no')
        ),
        windowed_trades AS (
            SELECT
                m.event_ticker,
                m.team,
                m.result,
                m.outcome,
                m.close_time,
                t.yes_price_dollars,
                t.count_fp,
                t.count_fp * t.yes_price_dollars       AS trade_value
            FROM read_parquet('kxnbagame_trades_all.parquet') t
            INNER JOIN markets m ON t.market_ticker = m.ticker
            WHERE CAST(t.created_time AS TIMESTAMPTZ)
                      BETWEEN m.close_time - INTERVAL '210 minutes'
                          AND m.close_time - INTERVAL '150 minutes'
        ),
        per_market AS (
            SELECT
                event_ticker,
                team,
                result,
                outcome,
                COUNT(*)                                                AS num_trades,
                SUM(count_fp)                                           AS total_contracts,
                SUM(trade_value) / NULLIF(SUM(count_fp), 0)            AS wavg_yes_price
            FROM windowed_trades
            GROUP BY event_ticker, team, result, outcome
        )
        -- one row per game: winner's implied prob vs loser's implied prob
        SELECT
            event_ticker,
            MAX(CASE WHEN outcome = 1 THEN team END)               AS winner,
            MAX(CASE WHEN outcome = 0 THEN team END)               AS loser,
            ROUND(MAX(CASE WHEN outcome = 1 THEN wavg_yes_price END), 4) AS winner_implied_prob,
            ROUND(MAX(CASE WHEN outcome = 0 THEN wavg_yes_price END), 4) AS loser_implied_prob,
            SUM(CASE WHEN outcome = 1 THEN num_trades  ELSE 0 END) AS winner_trades,
            SUM(CASE WHEN outcome = 0 THEN num_trades  ELSE 0 END) AS loser_trades,
            ROUND(SUM(CASE WHEN outcome = 1 THEN total_contracts ELSE 0 END), 2) AS winner_contracts,
            ROUND(SUM(CASE WHEN outcome = 0 THEN total_contracts ELSE 0 END), 2) AS loser_contracts
        FROM per_market
        GROUP BY event_ticker
        ORDER BY event_ticker
    """).df()

wavg.head(20)
